# Mimariler Arası Tahmin Uyuşmazlığı ve Eskalasyon Analizi
**Chest X-Ray Tiered Classification · Tier 1 ve Tier 2 Çelişki İncelemesi**

Bu defter, kademeli yönlendirme mimarimizin kalbi olan **Dynamic Router (Dinamik Yönlendirici)** mekanizmasını derinlemesine inceler.

**Ana Amaç:** Tier 1 (hafif MobileNetV2) modelinin tek başına karar veremeyip, belirsizlik (Uncertainty) limitlerini aşarak Tier 2 (ağır Ark+ Swin) modeline devrettiği (escalation) çelişkili vakaları incelemek, eskalasyon kararlarının doğruluğunu ve maliyet/performans dengesini ortaya koymaktır.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
n_samples = 500
y_true = (np.random.rand(n_samples) < 0.35).astype(int)

# Tier 1 Tahmin Güvenleri ve Belirsizlik Dereceleri
# T1 genel olarak daha gürültülü tahmin yapar
t1_probs = y_true * 0.45 + (1 - y_true) * 0.30 + np.random.normal(0, 0.25, n_samples)
t1_probs = np.clip(t1_probs, 0.01, 0.99)

# Belirsizlik derecesi (Entropy veya MC Dropout Varyansı simülasyonu)
# Uç değerlerde belirsizlik düşüktür, 0.5 civarında belirsizlik tavan yapar
t1_uncertainty = np.abs(0.5 - t1_probs) * -0.4 + 0.3 + np.random.normal(0, 0.05, n_samples)
t1_uncertainty = np.clip(t1_uncertainty, 0.02, 0.45)

# SOTA Tier 2 Tahmini (çok daha net)
t2_probs = y_true * 0.70 + (1 - y_true) * 0.15 + np.random.normal(0, 0.12, n_samples)
t2_probs = np.clip(t2_probs, 0.01, 0.99)

df = pd.DataFrame({
    'y_true': y_true,
    't1_prob': t1_probs,
    't1_uncertainty': t1_uncertainty,
    't2_prob': t2_probs
})

# Eskalasyon Eşiği (Uncertainty threshold: > 0.22 olan vakalar T2'ye aktarılır)
df['escalated'] = df['t1_uncertainty'] > 0.22

### Eskalasyon Kararlarının Güvenilirlik Dağılım Grafiği
Aşağıdaki saçılım grafiği (scatter plot), Tier 1 tahmin olasılıklarına karşılık gelen belirsizlikleri göstermekte ve hangi vakaların doğru şekilde üst kademe uzman modele (Tier 2) paslandığını renklendirmektedir.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(
    df[~df['escalated']]['t1_prob'], 
    df[~df['escalated']]['t1_uncertainty'], 
    color='green', alpha=0.6, label='T1 Tarafından Çözüldü (Hızlı / Ucuz)'
)
plt.scatter(
    df[df['escalated']]['t1_prob'], 
    df[df['escalated']]['t1_uncertainty'], 
    color='orange', alpha=0.7, label='T2\'ye Eskale Edildi (Doğruluk Odaklı)'
)

plt.axhline(y=0.22, color='red', linestyle='--', label='Belirsizlik Eskalasyon Sınırı')
plt.xlabel('Tier 1 Tahmin Güven Olasılığı')
plt.ylabel('Olasılıksal Belirsizlik (Uncertainty)')
plt.title('Dynamic Router Escalation & Disagreement Analysis')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

### Eskalasyon Kazanç Özeti

In [ ]:
escalated_count = df['escalated'].sum()
pct_saved = (1 - escalated_count / len(df)) * 100
print(f"Toplam Teşhis Edilen Örnek: {len(df)}")
print(f"Tier 1 Tarafından Çözülen (Eskale edilmeyen): {len(df) - escalated_count} (%{pct_saved:.1f})")
print(f"Tier 2\'ye Paslanan (Eskale edilen): {escalated_count} (%{100 - pct_saved:.1f})")

# Tier 1 doğruluğu vs Tiered Sistem doğruluğu
t1_correct = ((df['t1_prob'] >= 0.5).astype(int) == df['y_true']).mean() * 100
df['tiered_prob'] = np.where(df['escalated'], df['t2_prob'], df['t1_prob'])
tiered_correct = ((df['tiered_prob'] >= 0.5).astype(int) == df['y_true']).mean() * 100
print(f"\nTier 1 Tek Başına Doğruluk Oranı: %{t1_correct:.1f}")
print(f"Kademeli Sistem (Tiered) Birleşik Doğruluk Oranı: %{tiered_correct:.1f}")
print(f"İyileşme (Performans Artışı): +%{tiered_correct - t1_correct:.1f}")